In [ ]:

# ============================================================
# INSURANCE CHARGES PREDICTION - COMPLETE MACHINE LEARNING PROJECT
# ============================================================

# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

print("=" * 65)
print("INSURANCE CHARGES PREDICTION PROJECT")
print("=" * 65)


# 2. DATA COLLECTION AND UNDERSTANDING
# ============================================================

print("\n1. DATA COLLECTION AND UNDERSTANDING")

df = pd.read_csv("insurance.csv")

print("\nFirst 5 Records:")
print(df.head())

print("\nLast 5 Records:")
print(df.tail())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())


# 3. DATA CLEANING
# ============================================================

print("\n2. DATA CLEANING")

# Remove duplicate records
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

# Handle missing values
# Numerical missing values will be handled using median imputation
# Categorical missing values will be handled using most frequent imputation
# inside the machine learning preprocessing pipeline.

print("\nMissing values after cleaning:")
print(df.isnull().sum())

# Separate numerical and categorical columns
numerical_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

target = "charges"

# Check expected columns
required_columns = numerical_features + categorical_features + [target]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")


# 4. EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

print("\n3. EXPLORATORY DATA ANALYSIS")

print("\nNumerical Features:")
print(df[numerical_features].describe())

print("\nCategorical Feature Counts:")

for column in categorical_features:
    print(f"\n{column.upper()}")
    print(df[column].value_counts())

# Distribution of numerical features
df[numerical_features + [target]].hist(
    figsize=(12, 9),
    bins=20,
    edgecolor="black"
)

plt.suptitle("Distribution of Numerical Features")
plt.tight_layout()
plt.show()

# Categorical feature distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, column in zip(axes, categorical_features):
    sns.countplot(data=df, x=column, ax=ax)
    ax.set_title(f"{column} Distribution")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

# Age vs charges
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="age",
    y="charges",
    hue="smoker"
)
plt.title("Age vs Insurance Charges")
plt.show()

# BMI vs charges
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="bmi",
    y="charges",
    hue="smoker"
)
plt.title("BMI vs Insurance Charges")
plt.show()

# Smoker vs charges
plt.figure(figsize=(7, 5))
sns.boxplot(
    data=df,
    x="smoker",
    y="charges"
)
plt.title("Smoking Status vs Insurance Charges")
plt.show()

# Region vs charges
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="region",
    y="charges"
)
plt.title("Region vs Insurance Charges")
plt.show()

# Outlier visualization
plt.figure(figsize=(12, 4))

for i, column in enumerate(["age", "bmi", "charges"], 1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=df[column])
    plt.title(f"{column} Outliers")

plt.tight_layout()
plt.show()

# Numerical correlation
plt.figure(figsize=(8, 6))
sns.heatmap(
    df.select_dtypes(include=np.number).corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Numerical Feature Correlation")
plt.show()

# Average charges by category
print("\nAverage Insurance Charges by Smoking Status:")
print(df.groupby("smoker")["charges"].mean())

print("\nAverage Insurance Charges by Gender:")
print(df.groupby("sex")["charges"].mean())

print("\nAverage Insurance Charges by Region:")
print(df.groupby("region")["charges"].mean())

print("\nAverage Insurance Charges by Number of Children:")
print(df.groupby("children")["charges"].mean())


# 5. FEATURE ENGINEERING AND PREPROCESSING
# ============================================================

print("\n4. FEATURE ENGINEERING AND PREPROCESSING")

# X = independent/input features
# y = dependent/target variable

X = df.drop(columns=[target])
y = df[target]

print("\nInput Features:")
print(X.head())

print("\nTarget Variable:")
print(y.head())

# Split numerical and categorical columns
numerical_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

# Numerical preprocessing:
# 1. Fill missing values using median
# 2. Scale numerical features using StandardScaler

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing:
# 1. Fill missing values using most frequent value
# 2. Convert categories into numerical columns using OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing methods

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

# Train-test split: 80% training and 20% testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=df["smoker"]  # Preserve smoker proportions in both partitions.
)

# Five-fold CV is used only on the training partition for model selection.
cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)


# 6. MODEL BUILDING
# ============================================================

print("\n5. MODEL BUILDING")

# Define regression algorithms

models = {
    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(alpha=1.0),

    "Lasso Regression": Lasso(alpha=0.1, max_iter=10000),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    )
}

# Optional XGBoost model
try:
    from xgboost import XGBRegressor

    models["XGBoost"] = XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        objective="reg:squarederror",
        verbosity=0
    )

    print("XGBoost is available.")

except ImportError:
    print("XGBoost is not installed. Other models will be trained.")
    print("To install it, run: python -m pip install xgboost")


# 7. TRAIN MODELS AND EVALUATE PERFORMANCE
# ============================================================

print("\n6. MODEL TRAINING AND EVALUATION")

results = []
trained_models = {}
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error"
}

for model_name, model in models.items():

    print(f"\nCross-validating: {model_name}")

    # Keep preprocessing inside the pipeline so each fold learns its own
    # imputers, scaler, and category encoder from that fold's training data.
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    cv_scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring,
        n_jobs=1, return_train_score=False
    )

    cv_r2 = cv_scores["test_r2"]
    cv_mae = -cv_scores["test_mae"]
    cv_mse = -cv_scores["test_mse"]
    cv_rmse = np.sqrt(cv_mse)

    results.append({
        "Model": model_name,
        "CV R2 Mean": cv_r2.mean(),
        "CV R2 Std": cv_r2.std(ddof=1),
        "CV MAE Mean": cv_mae.mean(),
        "CV MSE Mean": cv_mse.mean(),
        "CV RMSE Mean": cv_rmse.mean(),
        "CV RMSE Std": cv_rmse.std(ddof=1)
    })

    # Keep a fitted training-only pipeline for tuning the selected model.
    trained_models[model_name] = pipeline.fit(X_train, y_train)

    print(f"CV R2: {cv_r2.mean():.4f} +/- {cv_r2.std(ddof=1):.4f}")
    print(f"CV MAE: ${cv_mae.mean():,.2f}")
    print(f"CV RMSE: ${cv_rmse.mean():,.2f} +/- ${cv_rmse.std(ddof=1):,.2f}")


# 8. MODEL COMPARISON
# ============================================================

print("\n7. MODEL COMPARISON")

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="CV R2 Mean",
    ascending=False
).reset_index(drop=True)

print("\nRegression Model Performance:")
print(results_df.to_string(index=False))

best_model_name = results_df.loc[0, "Model"]

print("\nBest Model Before Tuning:", best_model_name)

# Compare R2 scores
plt.figure(figsize=(11, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="CV R2 Mean"
)

plt.title("Five-Fold Cross-Validation - Mean R2")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Compare RMSE
plt.figure(figsize=(11, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="CV RMSE Mean"
)

plt.title("Five-Fold Cross-Validation - Mean RMSE")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


# 9. HYPERPARAMETER TUNING
# ============================================================

print("\n8. HYPERPARAMETER TUNING")

# Tune the best-performing model from the initial comparison

parameter_grids = {

    "Linear Regression": {},

    "Ridge Regression": {
        "model__alpha": [0.01, 0.1, 1, 10, 100]
    },

    "Lasso Regression": {
        "model__alpha": [0.001, 0.01, 0.1, 1, 10]
    },

    "Decision Tree": {
        "model__max_depth": [3, 5, 10, None],
        "model__min_samples_split": [2, 5, 10]
    },

    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_split": [2, 5]
    },

    "Gradient Boosting": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [2, 3, 4]
    }
}

if best_model_name == "XGBoost":
    parameter_grid = {
        "model__n_estimators": [100, 200],
        "model__max_depth": [2, 3, 5],
        "model__learning_rate": [0.01, 0.05, 0.1]
    }

else:
    parameter_grid = parameter_grids[best_model_name]

best_pipeline = trained_models[best_model_name]

if parameter_grid:

    grid_search = GridSearchCV(
        estimator=best_pipeline,
        param_grid=parameter_grid,
        cv=cv,
        scoring="r2",
        n_jobs=1
    )

    grid_search.fit(X_train, y_train)

    final_model = grid_search.best_estimator_
    tuned_cv_r2 = grid_search.best_score_

    print("\nBest Parameters:")
    print(grid_search.best_params_)

    print("\nBest Cross-Validation R2 Score:")
    print(grid_search.best_score_)

else:
    print("Selected model has no tuning parameters in this setup.")
    final_model = best_pipeline
    tuned_cv_r2 = results_df.loc[0, "CV R2 Mean"]


# 10. FINAL MODEL EVALUATION
# ============================================================

print("\n9. FINAL MODEL EVALUATION")

final_predictions = final_model.predict(X_test)

final_r2 = r2_score(y_test, final_predictions)
final_mae = mean_absolute_error(y_test, final_predictions)
final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)

print("\nFinal Model:", best_model_name)

print("Final R2 Score:", round(final_r2, 4))
print("Final MAE:", round(final_mae, 2))
print("Final MSE:", round(final_mse, 2))
print("Final RMSE:", round(final_rmse, 2))

# Bootstrap confidence interval communicates uncertainty in holdout MAE.
rng = np.random.default_rng(42)
absolute_errors = np.abs(y_test.to_numpy() - final_predictions)
bootstrap_mae = np.array([
    rng.choice(absolute_errors, size=len(absolute_errors), replace=True).mean()
    for _ in range(2000)
])
mae_ci_low, mae_ci_high = np.percentile(bootstrap_mae, [2.5, 97.5])
print(f"95% bootstrap CI for holdout MAE: ${mae_ci_low:,.2f} to ${mae_ci_high:,.2f}")

# Actual vs Predicted visualization

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    final_predictions,
    alpha=0.7
)

plt.xlabel("Actual Insurance Charges")
plt.ylabel("Predicted Insurance Charges")
plt.title("Actual vs Predicted Insurance Charges")

# Ideal prediction line
minimum = min(y_test.min(), final_predictions.min())
maximum = max(y_test.max(), final_predictions.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.tight_layout()
plt.show()

# Residual analysis

residuals = y_test - final_predictions

plt.figure(figsize=(8, 5))

sns.histplot(
    residuals,
    kde=True,
    bins=30
)

plt.title("Residual Distribution")
plt.xlabel("Prediction Error")
plt.show()

plt.figure(figsize=(8, 5))

plt.scatter(
    final_predictions,
    residuals,
    alpha=0.7
)

plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Charges")
plt.show()


# 11. MODEL INTERPRETATION - FEATURE IMPORTANCE
# ============================================================

print("\n10. MODEL INTERPRETATION")

# Prefer native importance for tree estimators. For other estimators,
# permutation importance measures the holdout R2 decrease after shuffling.
estimator = final_model.named_steps["model"]
importance_method = "Native tree feature importance"

if hasattr(estimator, "feature_importances_"):
    transformed_names = final_model.named_steps["preprocessor"].get_feature_names_out()
    native_importance = pd.DataFrame({
        "Transformed Feature": transformed_names,
        "Importance": estimator.feature_importances_
    })

    def get_original_feature(transformed_name):
        name = transformed_name.split("__", 1)[-1]
        if transformed_name.startswith("cat__"):
            return name.split("_", 1)[0]
        return name

    native_importance["Feature"] = native_importance["Transformed Feature"].map(
        get_original_feature
    )
    importance_df = (
        native_importance.groupby("Feature", as_index=False)["Importance"]
        .sum()
    )
    importance_df["Importance Std"] = 0.0
else:
    importance_method = "Permutation importance (holdout R2 decrease)"
    importance_result = permutation_importance(
        final_model,
        X_test,
        y_test,
        n_repeats=20,
        random_state=42,
        scoring="r2",
        n_jobs=1
    )
    importance_df = pd.DataFrame({
        "Feature": X_test.columns,
        "Importance": importance_result.importances_mean,
        "Importance Std": importance_result.importances_std
    })

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(f"\nFeature Importance ({importance_method}):")
print(importance_df.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.barh(
    importance_df["Feature"],
    importance_df["Importance"],
    xerr=importance_df["Importance Std"],
    capsize=3
)
plt.gca().invert_yaxis()
plt.xlabel("Aggregated feature importance")
plt.title(f"Feature Importance ? {importance_method}")
plt.tight_layout()
plt.show()


# 12. SAMPLE INSURANCE CHARGE PREDICTION
# ============================================================

print("\n11. SAMPLE INSURANCE CHARGE PREDICTION")

new_customer = pd.DataFrame({
    "age": [30],
    "sex": ["female"],
    "bmi": [25.5],
    "children": [1],
    "smoker": ["no"],
    "region": ["southeast"]
})

predicted_charge = final_model.predict(new_customer)

print("\nNew Customer Details:")
print(new_customer)

print("\nPredicted Insurance Charges:")
print(round(predicted_charge[0], 2))


# 13. SAVE FINAL MODEL
# ============================================================

print("\n12. SAVING FINAL MODEL")

joblib.dump(
    final_model,
    "insurance_charges_model.pkl"
)

print("Final model saved successfully!")
print("File: insurance_charges_model.pkl")


# 14. BUSINESS INSIGHTS
# ============================================================

print("\n13. BUSINESS INSIGHTS")

smoker_average = df.groupby("smoker")["charges"].mean()

print("\nAverage Charges for Non-Smokers:",
      round(smoker_average.get("no", np.nan), 2))

print("Average Charges for Smokers:",
      round(smoker_average.get("yes", np.nan), 2))

if "yes" in smoker_average and "no" in smoker_average:
    ratio = smoker_average["yes"] / smoker_average["no"]
    print("Smoker to Non-Smoker Average Charge Ratio:",
          round(ratio, 2))

print("\nAverage Charges by Age:")
print(df.groupby("age")["charges"].mean().head(10))

print("\nMost Important Model Features:")
print(importance_df.head(3).to_string(index=False))


# 15. FINAL PROJECT SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FINAL PROJECT SUMMARY")
print("=" * 65)

print("Dataset Records Used:", len(df))
print("Input Features:", X.shape[1])
print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))
print("Models Compared:", len(models))
print("Best Model (selected by training CV):", best_model_name)
print("Baseline Best-Model CV R2 Mean:", round(results_df.loc[0, "CV R2 Mean"], 4))
print("Tuned Model CV R2 Score:", round(tuned_cv_r2, 4))
print("Final Holdout R2 Score:", round(final_r2, 4))
print("Final Holdout MAE:", round(final_mae, 2))
print(f"Holdout MAE 95% CI: ${mae_ci_low:,.2f} to ${mae_ci_high:,.2f}")
print("Final Holdout RMSE:", round(final_rmse, 2))

# Save actual run metrics for the Streamlit About page.
evaluation_payload = {
    "best_model": str(best_model_name),
    "selection_method": "Shuffled five-fold cross-validation on the training partition; final metrics use a separate holdout test partition.",
    "dataset_records": int(len(df)),
    "train_records": int(len(X_train)),
    "test_records": int(len(X_test)),
    "trained_models": [str(name) for name in models.keys()],
    "baseline_cv_r2": float(results_df.loc[0, "CV R2 Mean"]),
    "tuned_cv_r2": float(tuned_cv_r2),
    "test_metrics": {
        "R2": float(final_r2),
        "MAE": float(final_mae),
        "MSE": float(final_mse),
        "RMSE": float(final_rmse)
    },
    "mae_95_ci": [float(mae_ci_low), float(mae_ci_high)],
    "comparison": results_df.to_dict(orient="records"),
    "feature_importance_method": str(importance_method),
    "feature_importance": importance_df.to_dict(orient="records")
}

with open("model_evaluation.json", "w", encoding="utf-8") as evaluation_file:
    json.dump(evaluation_payload, evaluation_file, indent=2)

print("\nModel evaluation and interpretation saved to model_evaluation.json")
